# CAP and PACELC, in numbers
Companion notebook for *System Design, Plainly — Part 5: CAP and PACELC*.
Every simulation in the post runs here. Seeds are fixed, so re-running reproduces the exact numbers.


## 1. Quorum math: why W + R > N means never stale
N = 3 replicas. A write touches a random W-set, a read a random R-set. If the two sets
don't overlap, the read can miss the latest write. Run 20,000 trials per level.


In [ ]:
import random

def stale_rate(W, R, N=3, trials=20000, seed=11):
    rng = random.Random(seed)
    stale = 0
    for _ in range(trials):
        wset = set(rng.sample(range(N), W))
        rset = set(rng.sample(range(N), R))
        if not (wset & rset):
            stale += 1
    return stale / trials

for W, R in [(1, 1), (1, 2), (2, 2), (3, 1), (1, 3), (2, 1)]:
    print(f"W={W} R={R}  W+R={W + R}  stale={stale_rate(W, R):.3f}")

# W+R>N guarantees overlap, so zero stale reads -- check it holds exactly.
assert stale_rate(2, 2) == 0.0
assert stale_rate(3, 1) == 0.0
assert stale_rate(1, 3) == 0.0
# And the non-quorum levels really do go stale a third to two-thirds of the time.
assert stale_rate(1, 1) > 0.6
assert 0.3 < stale_rate(1, 2) < 0.4
print("quorum rule verified: W+R>N -> 0.000 stale in 20,000 trials")


## 2. The partition: AP diverges, CP rejects
100 writes land during a 60-second partition, split across two sides.
A key written on *both* sides is a conflict you must resolve at merge time.
Watch how the conflict rate collapses as the keyspace grows: hot keys are the danger, not volume.


In [ ]:
import random

def partition_sim(n_writes=100, keys=20, seed=11):
    rng = random.Random(seed)
    side_a, side_b = {}, {}
    for _ in range(n_writes):
        k = rng.randrange(keys)
        if rng.random() < 0.5:
            side_a[k] = side_a.get(k, 0) + 1
        else:
            side_b[k] = side_b.get(k, 0) + 1
    conflicts = sum(1 for k in side_a if k in side_b)
    return len(side_a), len(side_b), conflicts

for keys in (20, 100, 1000):
    a, b, c = partition_sim(keys=keys)
    print(f"{keys:>5} keys: side A touched {a}, side B {b}, "
          f"conflicting keys {c} ({c / keys * 100:.1f}% of keys)")

# CP side: the minority refuses writes instead of diverging.
rng = random.Random(11)
rejected = sum(1 for _ in range(100) if rng.random() < 0.5)
print(f"CP: minority side rejects {rejected}% of writes (0 conflicts, 0 divergence)")

assert partition_sim(keys=20)[2] == 13   # 65% of the 20 hot keys conflict
assert partition_sim(keys=1000)[2] == 1  # same 100 writes, wide keyspace: ~no conflicts


## 3. The E in PACELC: every consistency level is a round trip
Model: one RTT to the quorum + 0.5 ms to apply locally.
Same-AZ RTT ~0.5 ms, cross-AZ ~1-2 ms, US coast-to-coast ~60-70 ms, US-EU ~90 ms.


In [ ]:
modes = [
    ("Local ack (W=1, async repl.)", 1.0),
    ("Same-AZ quorum (W=2 of 3, 1 AZ)", 2.5),
    ("Cross-AZ quorum (W=2 of 3, 3 AZs)", 5.0),
    ("Cross-region quorum (US coast to coast)", 70.0),
    ("Global sync commit (Spanner-style, 3 regions)", 105.0),
]
for name, ms in modes:
    print(f"{name:<45} {ms:>6.1f} ms")
print(f"cross-region quorum is {70.0 / 5.0:.0f}x a same-AZ quorum; "
      f"global sync is {105.0 / 1.0:.0f}x a local ack")
assert 70.0 / 5.0 == 14.0 and 105.0 / 1.0 == 105.0


## 4. Tunable consistency: the same system, two answers
DynamoDB-style: eventual reads are free (local replica), strongly consistent reads
pay a quorum round trip. Simulate 10,000 reads against a replica that lags 200 ms
behind the primary on 5% of reads: how often does an eventual read see stale data
vs a strong read?


In [ ]:
import random

rng = random.Random(11)
N = 10000
lagging = 0.05          # 5% of reads hit a replica that hasn't received the latest write
stale_eventual = sum(1 for _ in range(N) if rng.random() < lagging)
stale_strong = 0        # strong read waits for the quorum: always current
print(f"eventual reads stale: {stale_eventual}/{N} = {stale_eventual / N:.1%}")
print(f"strong reads stale:   {stale_strong}/{N} = {stale_strong / N:.1%}")
print(f"price of strong: one extra quorum round trip per read "
      f"(~2.5 ms same-AZ, ~70 ms cross-region -- see section 3)")
assert 400 < stale_eventual < 600
assert stale_strong == 0


## 5. The decision rule, as a function
Pick per operation, not per system: money and uniqueness need C; feeds, carts and
counters want L; and the partition branch is decided by what a wrong answer costs.


In [ ]:
def choose(op, wrong_answer_cost, needs_fresh_read):
    """Return (partition choice, normal choice) for one operation."""
    p = "C (refuse rather than lie)" if wrong_answer_cost == "high" else "A (answer, reconcile later)"
    e = "C (wait for quorum)" if needs_fresh_read else "L (local read, replicate async)"
    return p, e

for op, cost, fresh in [
    ("charge a card", "high", True),
    ("add to cart", "low", False),
    ("like counter", "low", False),
    ("seat reservation", "high", True),
    ("user profile read", "low", False),
]:
    p, e = choose(op, cost, fresh)
    print(f"{op:<20} partition: {p:<28} normal: {e}")

assert choose("charge a card", "high", True) == (
    "C (refuse rather than lie)", "C (wait for quorum)")
assert choose("like counter", "low", False) == (
    "A (answer, reconcile later)", "L (local read, replicate async)")
